# Geração de Dados Sintéticos — `raw.areas`

**Objetivo:** Popular a tabela `raw.areas` com 20 registros sintéticos.

**Regras de integridade:**
- Os códigos de área devem incluir obrigatoriamente os 8 codes referenciados na tabela fato `raw.transacoes_financeiras`: `COM`, `COMP`, `FIN`, `JUR`, `MKT`, `OP`, `RH`, `TI`.
- `id_area_raw` é inteiro sequencial (1–20).
- `codigo_area` é o campo que referencia a tabela fato.

**Reprodutibilidade:** `seed = 42`

In [1]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ============================================================
import pandas as pd
import numpy as np
import hashlib
import uuid
import os
from datetime import datetime, timedelta
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

QTD_AREAS = 20
SOURCE_SYSTEM = 'ERP_CORPORATIVO'
SOURCE_ENTITY = 'departamentos'
INGESTION_ID = str(uuid.uuid4())
INGESTION_TS = datetime(2026, 1, 10, 8, 0, 0).strftime('%Y-%m-%dT%H:%M:%S.000Z')

print(f'ingestion_id: {INGESTION_ID}')
print(f'ingestion_ts: {INGESTION_TS}')

ingestion_id: 05dd8968-d6f2-46ff-b4c3-aa02b3c1e229
ingestion_ts: 2026-01-10T08:00:00.000Z


In [2]:
# ============================================================
# 2. DEFINIÇÃO DAS ÁREAS
# ============================================================
# Os 8 códigos abaixo são obrigatórios — referenciados na tabela fato.
# Mais 12 complementam o total de 20.

AREAS = [
    # --- Obrigatórias (referenciadas na tabela fato) ---
    {'codigo_area': 'COM',  'nome_area': 'Comercial',                  'gestor_responsavel': 'Lucas Pereira',    'email_gestor': 'lucas.pereira@empresa.com'},
    {'codigo_area': 'COMP', 'nome_area': 'Compliance',                 'gestor_responsavel': 'Daniela Souza',    'email_gestor': 'daniela.souza@empresa.com'},
    {'codigo_area': 'FIN',  'nome_area': 'Financeiro',                 'gestor_responsavel': 'Ana Oliveira',     'email_gestor': 'ana.oliveira@empresa.com'},
    {'codigo_area': 'JUR',  'nome_area': 'Jurídico',                   'gestor_responsavel': 'Patrícia Mendes',  'email_gestor': 'patricia.mendes@empresa.com'},
    {'codigo_area': 'MKT',  'nome_area': 'Marketing',                  'gestor_responsavel': 'Juliana Costa',    'email_gestor': 'juliana.costa@empresa.com'},
    {'codigo_area': 'OP',   'nome_area': 'Operações',                  'gestor_responsavel': 'Sandra Nunes',     'email_gestor': 'sandra.nunes@empresa.com'},
    {'codigo_area': 'RH',   'nome_area': 'Recursos Humanos',           'gestor_responsavel': 'Fernanda Lima',    'email_gestor': 'fernanda.lima@empresa.com'},
    {'codigo_area': 'TI',   'nome_area': 'Tecnologia da Informação',   'gestor_responsavel': 'Ricardo Almeida',  'email_gestor': 'ricardo.almeida@empresa.com'},
    # --- Complementares ---
    {'codigo_area': 'ADM',  'nome_area': 'Administração',              'gestor_responsavel': 'Carlos Silva',     'email_gestor': 'carlos.silva@empresa.com'},
    {'codigo_area': 'CTB',  'nome_area': 'Contabilidade',              'gestor_responsavel': 'Ricardo Ferreira', 'email_gestor': 'ricardo.ferreira@empresa.com'},
    {'codigo_area': 'LOG',  'nome_area': 'Logística',                  'gestor_responsavel': 'Paulo Rodrigues',  'email_gestor': 'paulo.rodrigues@empresa.com'},
    {'codigo_area': 'CPT',  'nome_area': 'Compras',                    'gestor_responsavel': 'Camila Rocha',     'email_gestor': 'camila.rocha@empresa.com'},
    {'codigo_area': 'PJT',  'nome_area': 'Gestão de Projetos',         'gestor_responsavel': 'Beatriz Alves',    'email_gestor': 'beatriz.alves@empresa.com'},
    {'codigo_area': 'QAS',  'nome_area': 'Qualidade e Auditoria',      'gestor_responsavel': 'André Santos',     'email_gestor': 'andre.santos@empresa.com'},
    {'codigo_area': 'ATD',  'nome_area': 'Atendimento ao Cliente',     'gestor_responsavel': 'Carla Silva',      'email_gestor': 'carla.silva@empresa.com'},
    {'codigo_area': 'VND',  'nome_area': 'Vendas',                     'gestor_responsavel': 'Roberto Santos',   'email_gestor': 'roberto.santos@empresa.com'},
    {'codigo_area': 'PES',  'nome_area': 'Pesquisa e Desenvolvimento', 'gestor_responsavel': 'Marcos Oliveira',  'email_gestor': 'marcos.oliveira@empresa.com'},
    {'codigo_area': 'PRD',  'nome_area': 'Produto',                    'gestor_responsavel': 'Felipe Martins',   'email_gestor': 'felipe.martins@empresa.com'},
    {'codigo_area': 'SEC',  'nome_area': 'Segurança Patrimonial',      'gestor_responsavel': 'Jorge Lima',       'email_gestor': 'jorge.lima@empresa.com'},
    {'codigo_area': 'ESG',  'nome_area': 'Sustentabilidade (ESG)',     'gestor_responsavel': 'Marina Costa',     'email_gestor': 'marina.costa@empresa.com'},
]

assert len(AREAS) == QTD_AREAS, f'Esperado {QTD_AREAS} áreas, encontrado {len(AREAS)}'
print(f'Total de áreas definidas: {len(AREAS)}')

Total de áreas definidas: 20


In [3]:
# ============================================================
# 3. CONSTRUÇÃO DO DATAFRAME + METADADOS
# ============================================================

def gerar_hash(row_dict):
    """Gera SHA-256 hash dos campos de negócio de uma linha."""
    campos = ['id_area_raw', 'codigo_area', 'nome_area', 'gestor_responsavel', 'email_gestor']
    conteudo = '|'.join(str(row_dict.get(c, '')) for c in campos)
    return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()

registros = []
for seq, area in enumerate(AREAS, start=1):
    row = {
        'id_area_raw':          seq,
        'codigo_area':          area['codigo_area'],
        'nome_area':            area['nome_area'],
        'gestor_responsavel':   area['gestor_responsavel'],
        'email_gestor':         area['email_gestor'],
        'ingestion_id':         INGESTION_ID,
        'ingestion_ts':         INGESTION_TS,
        'source_system':        SOURCE_SYSTEM,
        'source_entity':        SOURCE_ENTITY,
        'row_seq':              seq,
        'raw_row_hash':         gerar_hash({'id_area_raw': seq, **area}),
    }
    registros.append(row)

df_areas = pd.DataFrame(registros)
print(f'Shape: {df_areas.shape}')
df_areas.head()

Shape: (20, 11)


,id_area_raw,codigo_area,nome_area,gestor_responsavel,email_gestor,ingestion_id,ingestion_ts,source_system,source_entity,row_seq,raw_row_hash
0,1,COM,Comercial,Lucas Pereira,lucas.pereira@empresa.com,05dd8968-d6f2-46ff-b4c3-aa02b3c1e229,2026-01-10T08:00:00.000Z,ERP_CORPORATIVO,departamentos,1,1893f202b0735232be7f7f9318bdad302113c1f74d24d7...
1,2,COMP,Compliance,Daniela Souza,daniela.souza@empresa.com,05dd8968-d6f2-46ff-b4c3-aa02b3c1e229,2026-01-10T08:00:00.000Z,ERP_CORPORATIVO,departamentos,2,1897f15ebf5c30011b434ac3b5420f826da9e44de95cb9...
2,3,FIN,Financeiro,Ana Oliveira,ana.oliveira@empresa.com,05dd8968-d6f2-46ff-b4c3-aa02b3c1e229,2026-01-10T08:00:00.000Z,ERP_CORPORATIVO,departamentos,3,2616c09aabef0b9daffa008a098ca9c8a768fcd35ef607...
3,4,JUR,Jurídico,Patrícia Mendes,patricia.mendes@empresa.com,05dd8968-d6f2-46ff-b4c3-aa02b3c1e229,2026-01-10T08:00:00.000Z,ERP_CORPORATIVO,departamentos,4,0a8d1ba1805cf6eb4899c56beecad58ad79cf9634ad59e...
4,5,MKT,Marketing,Juliana Costa,juliana.costa@empresa.com,05dd8968-d6f2-46ff-b4c3-aa02b3c1e229,2026-01-10T08:00:00.000Z,ERP_CORPORATIVO,departamentos,5,457f1037dcf85f29f487ab783912a072f64035207f3117...


In [ ]:
# ============================================================
# 4. VALIDAÇÕES
# ============================================================

CODIGOS_FATO = {'COM', 'COMP', 'FIN', 'JUR', 'MKT', 'OP', 'RH', 'TI'}
codigos_gerados = set(df_areas['codigo_area'])

assert CODIGOS_FATO.issubset(codigos_gerados), \
    f'Códigos ausentes: {CODIGOS_FATO - codigos_gerados}'

assert df_areas['id_area_raw'].nunique() == QTD_AREAS, 'IDs duplicados!'
assert df_areas['codigo_area'].nunique() == QTD_AREAS, 'Códigos duplicados!'
assert df_areas['raw_row_hash'].nunique() == QTD_AREAS, 'Hashes duplicados!'

# print('Todos os 8 códigos da tabela fato estão presentes.')
#p rint('Sem IDs duplicados.')
# print('Sem hashes duplicados.')

print(df_areas[['id_area_raw', 'codigo_area', 'nome_area']].to_string(index=False))

✔ Todos os 8 códigos da tabela fato estão presentes.
✔ Sem IDs duplicados.
✔ Sem hashes duplicados.

Distribuição de areas:
 id_area_raw codigo_area                  nome_area
           1         COM                  Comercial
           2        COMP                 Compliance
           3         FIN                 Financeiro
           4         JUR                   Jurídico
           5         MKT                  Marketing
           6          OP                  Operações
           7          RH           Recursos Humanos
           8          TI   Tecnologia da Informação
           9         ADM              Administração
          10         CTB              Contabilidade
          11         LOG                  Logística
          12         CPT                    Compras
          13         PJT         Gestão de Projetos
          14         QAS      Qualidade e Auditoria
          15         ATD     Atendimento ao Cliente
          16         VND                    

In [5]:
# ============================================================
# 5. EXPORTAÇÃO PARA CSV
# ============================================================

workspace = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
output_dir = os.path.join(workspace, 'data', 'raw', 'areas')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'areas.csv')
df_areas.to_csv(output_path, index=False, encoding='utf-8')

print(f'Arquivo exportado: {output_path}')
print(f'Total de registros: {len(df_areas)}')

Arquivo exportado: c:\Users\Adam\Documents\Repositorio\TCC\SCAP\data\raw\areas\areas.csv
Total de registros: 20
